In [2]:
import sys
# !pip install langchain-tavily

In [19]:
import os
os.environ['USER_AGENT'] = 'Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:47.0) Gecko/20100101 Firefox/47.0'
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import bs4
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_openai_tools_agent, tool, AgentExecutor
langchain.__version__


'0.3.27'

In [9]:
with open("data/secrets/key.secret") as f:
    os.environ["OPENAI_API_KEY"] = f.read()
with open("data/secrets/tavily.secret") as f:
    os.environ["TAVILY_API_KEY"] = f.read()

In [10]:
tool = TavilySearch(
    max_results=3,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)
tool.invoke({"query": "Quel météo fait il à Grenoble ?"})

{'query': 'Quel météo fait il à Grenoble ?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.tameteo.com/grenoble/aujourdhui',
   'title': "Météo Grenoble Aujourd'hui - tameteo.com | Meteored",
   'content': "Aujourd'hui à Grenoble, éclaircies ce matin, avec des températures moyennes autour de 13°C. Cet après-midi, couvert avec des températures d'environ 20°C.",
   'score': 0.85157084,
   'raw_content': None},
  {'url': 'https://www.neometeo.com/meteo-france/meteo-grenoble-aujourdhui-ville-14505',
   'title': "Quelle météo fait-il aujourd'hui et demain à Grenoble - NeoMeteo",
   'content': "La température minimum aujourd'hui à Grenoble sera de 3.2°. Quelle sera la vitesse du vent aujourd'hui à Grenoble ? Le vent soufflera entre 10.3",
   'score': 0.8443195,
   'raw_content': None},
  {'url': 'https://www.meteoart.com/europe/france/auvergne-rhone-alpes/grenoble?page=tomorrow',
   'title': 'Météo à Grenoble demain - MeteoArt.com',
   'conte

In [23]:
agent = create_react_agent(llm, [tavily_search_tool])

user_input = "What nation hosted the Euro 2024? Include only wikipedia sources."

for step in agent.stream(
    {"messages": user_input},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What nation hosted the Euro 2024? Include only wikipedia sources.
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_v5y3zD7KG6etsFAQWs6qf8xo)
 Call ID: call_v5y3zD7KG6etsFAQWs6qf8xo
  Args:
    query: Euro 2024 host nation
    include_domains: ['wikipedia.org']
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "Euro 2024 host nation", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://simple.wikipedia.org/wiki/UEFA_Euro_2024", "title": "UEFA Euro 2024", "content": "| Top scorer(s) | Harry Kane Georges Mikautadze Jamal Musiala Cody Gakpo Ivan Schranz\") Dani Olmo(3 goals each) | The 23 remaining spots will be determined through qualifying, with the group stage taking place from March to November 2023, and the play-offs li